# Chargement du modèle et du tokenizer

In [1]:
from model import Model

model = Model(max_content = 3)

h:\Documents\GitHub\NLP_narrative-game_CY-Tech_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 149.74it/s]


In [ ]:
system_content = "Tu es le Game Master d'un jeu de rôle fantasy en solo. Tu décris l'univers, les conséquences des actions du joueur et proposes toujours une liste d'actions possibles sous la forme A, B ou C. Le joueur doit obligatoirement choisir l'une des actions proposées. Si le joueur propose une action qui n'est pas dans la liste, indique clairement qu'elle est impossible et rappelle les actions disponibles. Les descriptions doivent être courtes, de 1 ou 2 phrases. "

# Paramètres

Le format de conversation attendu est le chat template mentionné dans le notebook d'observation du modèle: 
un content pour le role system, puis des succèssion de contents pour les roles "user" et "assistant". 

max_new_tokens = 256: cette longueur me semble nécessaire pour un agent jouant le rôle du game master, ni trop longue demandant trop de temps d'exécution <br>
do_sample = True: nous utilisons l'agent notamment pour sa créativité <br>
temperature = 1: nous utilisons la température par défaut, le hasard est un élément important, mais pas besoin de l'augmenter non plus pour que le modèle reste contrôlé <br>
top-k = 30: nous gardons les 30 tokens les plus probables pour un filtre correcte avec une certaine incertitude pour avoir des scénariots différents <br>
top-p = 0.8: nous filtrons les tokens les moins probables. 

# Test

## Zero-shot

### Action valide

In [ ]:
model.init_history(system_content)

In [26]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je choisi A")

model.generate()

print(model.assistant_content)

La porte s'ouvre lentement sur un vieil auberge dans le désert. Dans l'intérieur, vous apercevez un vieux baron tenant son marteau à deux manchettes. Il se redresse doucement de sa chaise et vous observe sans sourire. Il murmure : "Bienvenue à mon domaine... si je peux l'appeler ainsi." Il saisit votre épée et vous ordonne de rentrer chez-vous. Que feriez-vous ensuite ? A) Rencontrer le Baron pour discuter de votre arrivée B) Faire demi-tour et repartir immédiatement C) Dormir quelques jours dans votre nouveau quartier.


In [27]:
model.add_content("user", "Je choisi A")

model.generate()

print(model.assistant_content)

Le Baron s'incline et fait signe que vous pouvez entrer. Une pièce éclairée par la lumière orangée d'une lampe de chevet vous accueille. Sur une table basse, un verre d'eau glacée et une assiette pleine de biscuits dorés sont disposés. Vous sentez le confort, mais aussi l'hostilité sous ces gestes civils. Quel choix faisiez-vous ? A) Manger quelque chose avant d'enquêter B) Demander aux serviteurs de vous apporter un réconfort C) Rester là pour voir comment cela va se passer.


Le model a la réaction attendu, avec une liste des 3 choix ainsi qu'une succession logique des événements. 

### Action invalide

In [28]:
model.init_history(system_content)

In [29]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je fait demi tour. ")

model.generate()

print(model.assistant_content)

Bien joué! Le symbole de la lune semble le plus proche maintenant.


Le modèle ne contredit pas l'utilisateur quand il fait une action invalide. 

## Few-shot

## Action valide

In [37]:
exemples = """
Les exemples ci-dessous sont des exemples indépendants.
Ils servent uniquement à montrer le comportement attendu du Game Master.
Ils ne font PAS partie de la partie actuelle.

===== EXEMPLE 1 =====

Situation :
Le joueur se trouve dans une pièce avec une porte verrouillée.

Action du joueur :
Je cherche une sortie.

Réponse attendue :
Vous inspectez la pièce et remarquez une fenêtre ainsi qu'une porte verrouillée.

A. Examiner la fenêtre
B. Chercher une clé
C. Examiner la serrure


===== EXEMPLE 2 =====

Situation :
Le joueur est dans un bâtiment moderne.

Action du joueur :
Je lance un sort pour faire apparaître un dragon.

Réponse attendue :
Cette action n'est pas cohérente avec l'univers moderne.

A. Explorer le couloir
B. Examiner la porte
C. Chercher un objet utile


===== EXEMPLE 3 =====

Situation :
Le joueur n'a trouvé aucune clé.

Action du joueur :
J'utilise la clé que j'ai trouvée.

Réponse attendue :
Vous ne possédez actuellement aucune clé. Vous devez d'abord en trouver une.

A. Fouiller le bureau
B. Examiner la porte
C. Explorer la pièce
"""

In [38]:
model.init_history(system_content)

model.add_content("user", exemples)

In [39]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je choisi A")

model.generate()

print(model.assistant_content)

La lune illumine le chemin vers l'intérieur de votre maison. Une chaleur douce s'inverse autour de vous, rendant l'environnement accueillant et calme. Vous entendez des bruits de musique à travers le couloir, mais lorsque vous pénétracez sur la première pièce, elle devient désertée. La lumière s'éteint et une mélancolie se répand tout autour de vous.

Voici vos options actuelles :
A) Entrez dans cette pièce.
B) Revenez vers la porte principale.
C) Retournez directement à la cuisine.

Que choisirez-vous ?


Le model a la réaction attendu, avec une liste des 3 choix ainsi qu'une succession logique des événements. 

Action invalide

In [40]:
model.init_history(system_content)

model.add_content("user", exemples)

In [41]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je fait demi tour. ")

model.generate()

print(model.assistant_content)

Bien joué ! Quelque chose retient votre attention ?


Le modèle ne contredit pas l'utilisateur quand il fait une action invalide. 